In [1]:
import tweepy
import os
import pandas as pd
import time
from dotenv import load_dotenv

# Load API credentials
load_dotenv()

API_KEY = os.getenv("API_KEY")
API_SECRET = os.getenv("API_SECRET")
ACCESS_TOKEN = os.getenv("ACCESS_TOKEN")
ACCESS_SECRET = os.getenv("ACCESS_SECRET")
BEARER_TOKEN = os.getenv("BEARER_TOKEN")

# Authenticate with Twitter API
client = tweepy.Client(bearer_token=BEARER_TOKEN)

# Define query for Amazon tweets
query = "(Amazon OR @AmazonHelp OR #AmazonIndia) (issue OR complaint OR refund OR support) -is:retweet lang:en"

# Define storage path on remote server
server_path = "/workspace/Customer 360/tweets_data"
os.makedirs(server_path, exist_ok=True)

# File path for storing tweets
file_path = os.path.join(server_path, "amazon_tweets.csv")

# Function to fetch tweets
def fetch_tweets_api():
    try:
        tweets = client.search_recent_tweets(query=query, tweet_fields=['created_at', 'text'], max_results=100)
        tweet_data = [[tweet.created_at, tweet.text] for tweet in tweets.data]
        df = pd.DataFrame(tweet_data, columns=["Timestamp", "Tweet"])

        # Check if file exists to determine starting S. No.
        if os.path.exists(file_path):
            existing_data = pd.read_csv(file_path)
            start_index = len(existing_data) + 1
        else:
            start_index = 1

        # Add Serial Number Column
        df.insert(0, "S. No.", range(start_index, start_index + len(df)))

        # Append new tweets to CSV file
        df.to_csv(file_path, mode='a', header=not os.path.exists(file_path), index=False)

        print(f"{len(tweet_data)} tweets saved to {file_path}")

    except Exception as e:
        print(f"Error fetching tweets from API: {e}")

# Run script every 15 minutes
while True:
    fetch_tweets_api()
    print("Waiting for the next fetch cycle...")
    time.sleep(900)  # 900 seconds = 15 minutes


100 tweets saved to /workspace/Customer 360/tweets_data/amazon_tweets.csv
Waiting for the next fetch cycle...
Error fetching tweets from API: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Waiting for the next fetch cycle...
Error fetching tweets from API: Error tokenizing data. C error: Expected 2 fields in line 3, saw 3

Waiting for the next fetch cycle...
Error fetching tweets from API: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Waiting for the next fetch cycle...
Error fetching tweets from API: 429 Too Many Requests
Usage cap exceeded: Monthly product cap
Waiting for the next fetch cycle...
Error fetching tweets from API: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Waiting for the next fetch cycle...


KeyboardInterrupt: 

In [4]:
import snscrape.modules.twitter as sntwitter
import pandas as pd
import os
import re
import time

# Define storage path on remote server
server_path = "/workspace/Customer 360/tweets_data"
os.makedirs(server_path, exist_ok=True)

# Define file path
file_path = os.path.join(server_path, "amazon_tweets_cleaned.csv")

# Define search query with improved filtering
query = "(Amazon OR @AmazonHelp OR #AmazonIndia) (complaint OR problem OR refund OR return OR damaged OR delay OR missing OR scam OR fraud OR bad OR disappointed OR worst OR issue OR support OR service) \
         -filter:links -filter:replies -filter:retweets -filter:media -#deals -#offers -#ad lang:en since:2024-01-01"

# Function to clean tweets
def clean_tweet(text):
    text = re.sub(r"http\S+", "", text)  # Remove links
    text = re.sub(r"@\S+", "", text)  # Remove mentions
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)  # Remove special characters
    text = text.strip()  # Remove leading and trailing spaces
    return text

# Function to fetch tweets using SNSCRAPE
def fetch_tweets_scrape():
    try:
        tweets = []
        for i, tweet in enumerate(sntwitter.TwitterSearchScraper(query).get_items()):
            if i > 100:  # Fetch 100 per request (adjust as needed)
                break
            cleaned_tweet = clean_tweet(tweet.content)
            if len(cleaned_tweet) > 20:  # Exclude short junk tweets
                tweets.append([tweet.date, cleaned_tweet])

        df = pd.DataFrame(tweets, columns=["Timestamp", "Tweet"])

        # Handle S. No. (Check existing data for correct numbering)
        if os.path.exists(file_path):
            existing_data = pd.read_csv(file_path)
            start_index = len(existing_data) + 1
        else:
            start_index = 1

        df.insert(0, "S. No.", range(start_index, start_index + len(df)))

        # Append to CSV
        df.to_csv(file_path, mode='a', header=not os.path.exists(file_path), index=False)

        print(f"{len(df)} clean tweets saved to {file_path}")

    except Exception as e:
        print(f"Error fetching tweets: {e}")

# Run script every 15 minutes
while True:
    fetch_tweets_scrape()
    print("Waiting for the next fetch cycle...")
    time.sleep(900)  # 900 seconds = 15 minutes


AttributeError: 'FileFinder' object has no attribute 'find_module'

In [5]:
import snscrape.modules.twitter as sntwitter
print("snscrape is working!")


AttributeError: 'FileFinder' object has no attribute 'find_module'